# ARCH 6133 Places / Platforms
## NYC Site Selection: Algorithms Do Not Pick Sites, People Do

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danmillr/places-platforms/blob/main/tutorials/ARCH6133_NYC_Site_Selection.ipynb)

---

### What this notebook does

This notebook walks you through algorithmic **site selection** for New York City — the process of taking a set of criteria, applying them across thousands of candidate locations, and surfacing a shortlist of places that score well. You will:

- Re-fetch every dataset you need (this notebook is fully self-contained — it does not depend on files produced by `ARCH6133_NYC_Index_Builder.ipynb` or any other notebook).
- Aggregate point data to census tracts and compute the derived variables every scenario needs: heat vulnerability, transit access, library and health facility gaps, flood-zone status, city-owned lot share.
- Step through three pre-built example scenarios (cooling centers, street trees, library gap analysis), each with a fully exposed criteria table, weight rationale, and limitations note.
- Build your own scenario with hard feasibility constraints, criteria, directions, and weights — and produce a Methodology Card that documents every choice you made.
- Export the resulting candidate tracts as GeoJSON and CSV for downstream work.

### How this differs from the Index Builder notebook

| Index Builder | Site Selection |
|---|---|
| Asks: how does this place compare to others? | Asks: which places best fit this specific use? |
| Output is a continuous score for **every** unit | Output is a **shortlist** of candidates plus a score |
| No hard constraints — all variables are weights | Hard constraints (flood, lot ownership, population floor) eliminate tracts before scoring |
| Tells a story about a city | Proposes places where something could go |

Both are useful. They answer different questions. **Neither one chooses a site.** A person chooses a site, in a room with other people, after the algorithm has narrowed the field. Treat this notebook as a tool for narrowing, not for deciding.

### How to get a free Census API key

1. https://api.census.gov/data/key_signup.html — free, takes a minute, key arrives by email.
2. In Colab open the Secrets panel (key icon, left sidebar), add a secret named `CENSUS_API_KEY`, paste the key, toggle Notebook access ON.

### Data sources, update frequencies, and known limitations

| Source | Update Frequency | Watch out for |
|---|---|---|
| ACS 5-year (Census API) | Annually | High MOE at block group scale |
| PLUTO (NYC OD) | ~2x per year | `ownertype` and `landuse` fields can be missing |
| Street Tree Census 2015 (NYC OD) | 2015 snapshot | Streets only — parks and private trees absent |
| Wi-Fi Hotspots (NYC OD) | Quarterly | Tourist-heavy zones overcounted |
| Subway Entrances + Stations (**NY State** OD) | Periodic | ADA flag can lag actual elevator status |
| Heat Vulnerability Index (DOHMH) | Annual | Published at ZCTA scale, not NTA |
| 311 (NYC OD) | Daily | Reflects who calls, not where conditions are worst |
| GreenThumb Community Gardens (NYC OD) | Periodic | Dataset is marked archived; cross-check with DPR |
| NYC Facilities Database (NYC OD) | Quarterly | Source for both library and health facility filters |
| Future Floodplain 2020s (NYC OD) | Periodic | NYC's current 100-year-floodplain layer |

### Suitability vs. feasibility

This is the single most important distinction in the notebook. A **suitable** tract is one whose population and conditions match a proposed use well. A **feasible** site is one where the use can actually be built — there is a vacant building, the zoning permits it, the utility connections exist, the political will is present, the community is willing. Suitability is a property of places. Feasibility is a property of projects.

**This notebook computes suitability. It cannot compute feasibility.** Every output here is a starting point for a feasibility study, not a substitute for one.

### How to enable ipywidgets in Colab

Module 0 runs `output.enable_custom_widget_manager()` for you. If a slider or checkbox fails to render later, restart the runtime and re-run from the top of Module 0.

### Estimated runtime

Module 0: < 1 minute
Module 1: 8–12 minutes (this is the big fetch — every dataset, then spatial joins)
Module 2: < 1 minute
Module 3: 2–3 minutes (three example scenarios)
Module 4: depends on you
Module 5: < 1 minute

Total: ~15–20 minutes for an end-to-end run.


---

## Module 0 — Setup and Configuration

Run every cell in order. If your runtime resets later, come back here and re-run from the top — the Drive mount, package installs, API key handle, and helper functions defined here are used everywhere else in the notebook.

### Mount Google Drive

Everything this notebook saves — raw data, prepared GeoJSON, suitability maps, methodology cards — lives in a folder in your Drive.

In [ ]:
# Mount your personal Google Drive into the Colab filesystem.
from google.colab import drive
drive.mount('/content/drive')

### Install required packages

One consolidated install. Takes 60–90 seconds the first time.

In [ ]:
# One consolidated install (-q reduces dependency-resolution noise).
!pip install -q requests pandas geopandas folium mapclassify matplotlib seaborn scikit-learn ipywidgets tqdm census shapely

### Store your Census API key in Colab Secrets

NYC and NY State Open Data endpoints used in this notebook do **not** require authentication, but the US Census API does.

1. https://api.census.gov/data/key_signup.html — free
2. In Colab, click the key icon in the left sidebar (Secrets)
3. Add a secret named `CENSUS_API_KEY`, paste the key as the value, toggle Notebook access ON
4. Run the cell below

In [ ]:
# Pull the Census API key from Colab Secrets and validate.
from google.colab import userdata

CENSUS_API_KEY = userdata.get('CENSUS_API_KEY')

if not CENSUS_API_KEY:
    print("ERROR: CENSUS_API_KEY not found in Colab Secrets.")
    print("Follow the instructions above to add it, then re-run this cell.")
    raise ValueError("Missing CENSUS_API_KEY")
else:
    print("Census API key loaded. Open Data endpoints do not require a key.")

### Configure outputs

`TOP_N` controls how many candidate tracts every scenario surfaces.

In [ ]:
# Project configuration. Edit and re-run if you change defaults.
OUTPUT_FOLDER = '/content/drive/MyDrive/NYCSiteSelection/'
TOP_N = 15

import os

SUBFOLDERS = ['data', 'maps', 'exports', 'cards']
for sub in [''] + SUBFOLDERS:
    os.makedirs(os.path.join(OUTPUT_FOLDER, sub), exist_ok=True)

# Enable ipywidgets in Colab so Module 4 renders correctly.
from google.colab import output as colab_output
colab_output.enable_custom_widget_manager()

print(f"Setup complete. Output folder ready at {OUTPUT_FOLDER}.")
for sub in SUBFOLDERS:
    print(f"  - {sub}/")

### Reusable helpers

Four functions every later module reuses:
- `fetch_nyc_open_data(endpoint, params, max_rows, domain)` — paginates Socrata. The `domain` parameter lets us read from `data.ny.gov` when needed (the MTA subway dataset lives there).
- `fetch_census(variables, geography, year)` — pulls ACS for the five NYC counties and joins TIGER tract geometry.
- `buffer_points(gdf, meters)` — re-projects to NY State Plane (EPSG:2263, feet), buffers by the given distance in **meters**, returns geometry back in WGS84 (EPSG:4326). Important: lat/lng buffering in degrees is nonsensical at NYC's latitude — always project before you buffer.
- `compute_suitability(gdf, criteria_dict)` — normalizes, applies direction, weights, and sums the named columns; returns the input GeoDataFrame with a new `suitability_score` column.

In [ ]:
# Reusable helpers.
import requests
import pandas as pd
import geopandas as gpd
import numpy as np
from tqdm.auto import tqdm

NYC_COUNTIES = {
    "005": "Bronx",
    "047": "Kings",       # Brooklyn
    "061": "New York",    # Manhattan
    "081": "Queens",
    "085": "Richmond",    # Staten Island
}

def fetch_nyc_open_data(endpoint, params=None, max_rows=50000, page_size=5000,
                        domain="data.cityofnewyork.us"):
    """Page through a Socrata resource and return a DataFrame of all rows."""
    base_url = f"https://{domain}/resource/{endpoint}"
    params = dict(params or {})
    collected = []
    progress = tqdm(total=max_rows, desc=f"Fetching {endpoint}", unit="rows")
    for offset in range(0, max_rows, page_size):
        batch_limit = min(page_size, max_rows - offset)
        params["$limit"] = batch_limit
        params["$offset"] = offset
        try:
            response = requests.get(base_url, params=params, timeout=60)
            response.raise_for_status()
            batch = response.json()
        except Exception as fetch_error:
            print(f"  Fetch stopped at offset {offset}: {fetch_error}")
            break
        if not isinstance(batch, list) or len(batch) == 0:
            break
        collected.extend(batch)
        progress.update(len(batch))
        if len(batch) < batch_limit:
            break
    progress.close()
    return pd.DataFrame(collected)

# One-time per-session TIGER tract cache.
_TIGER_TRACTS_CACHE = {}

def _load_nyc_tract_geometry(year=2022):
    """Download and cache TIGER cartographic boundary tracts for NYC."""
    cache_key = ("tracts", year)
    if cache_key in _TIGER_TRACTS_CACHE:
        return _TIGER_TRACTS_CACHE[cache_key]
    url = f"https://www2.census.gov/geo/tiger/GENZ{year}/shp/cb_{year}_36_tract_500k.zip"
    print(f"Downloading TIGER tract geometry for NY State ({year})...")
    tracts = gpd.read_file(url)
    tracts = tracts[tracts["COUNTYFP"].isin(NYC_COUNTIES.keys())].copy()
    tracts["GEOID"] = tracts["GEOID"].astype(str)
    _TIGER_TRACTS_CACHE[cache_key] = tracts
    return tracts

def fetch_census(variables, geography="tract", year=2022):
    """Pull ACS 5-year estimates for NYC counties and attach tract geometry."""
    from census import Census
    census_client = Census(CENSUS_API_KEY, year=year)
    all_rows = []
    for county_fips in NYC_COUNTIES:
        try:
            county_rows = census_client.acs5.state_county_tract(
                variables, "36", county_fips, Census.ALL)
            all_rows.extend(county_rows)
        except Exception as census_error:
            print(f"Census fetch failed for county {county_fips}: {census_error}")
    df = pd.DataFrame(all_rows)
    if df.empty:
        print("Census API returned no rows. Check your API key.")
        return df
    df["GEOID"] = (df["state"].astype(str) + df["county"].astype(str) +
                   df["tract"].astype(str))
    tracts = _load_nyc_tract_geometry(year=year)
    merged = tracts.merge(df, on="GEOID", how="left")
    return merged

def buffer_points(point_gdf, meters):
    """Buffer points by distance in meters, projecting through EPSG:2263."""
    if point_gdf.empty:
        return point_gdf
    feet_per_meter = 3.28084
    in_state_plane = point_gdf.to_crs(epsg=2263)
    buffered = in_state_plane.copy()
    buffered["geometry"] = in_state_plane.geometry.buffer(meters * feet_per_meter)
    return buffered.to_crs(epsg=4326)

def compute_suitability(gdf, criteria_dict, normalizer="min_max"):
    """Normalize, direction-correct, weight, and sum criteria into a score.

    criteria_dict : list of dicts with keys: variable, weight, invert (bool).
    """
    normalizer_fn = NORMALIZERS[normalizer]
    score = pd.Series(np.zeros(len(gdf)), index=gdf.index)
    for criterion in criteria_dict:
        column = criterion["variable"]
        weight = float(criterion["weight"])
        invert = bool(criterion.get("invert", False))
        values = pd.to_numeric(gdf[column], errors="coerce")
        normalized = normalizer_fn(values).fillna(0.5)
        if invert:
            normalized = 1.0 - normalized
        score = score + weight * normalized
    out = gdf.copy()
    out["suitability_score"] = score
    return out

def normalize_min_max(series):
    """Map values to [0, 1] using the observed min and max."""
    minimum = series.min(skipna=True)
    maximum = series.max(skipna=True)
    if maximum == minimum:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - minimum) / (maximum - minimum)

def normalize_percentile(series):
    """Map values to [0, 1] by their rank percentile."""
    return series.rank(pct=True, method="average")

def normalize_zscore(series):
    """Center to mean 0, scale by std, then map to a [0, 1] window."""
    mu = series.mean(skipna=True)
    sigma = series.std(skipna=True)
    if sigma == 0 or pd.isna(sigma):
        return pd.Series(np.zeros(len(series)), index=series.index)
    return ((((series - mu) / sigma).clip(-3, 3)) + 3) / 6

NORMALIZERS = {"min_max":    normalize_min_max,
               "percentile": normalize_percentile,
               "zscore":     normalize_zscore}

print("Helpers ready: fetch_nyc_open_data, fetch_census, buffer_points, "
      "compute_suitability.")

---

## Module 1 — Data Fetch and Preparation

This is the biggest module. We pull every dataset the rest of the notebook needs, spatially join the point data to census tracts, and compute the derived variables every scenario calls. Budget 8–12 minutes for the fetch and joins to finish.

Before any code, read the opening critique below. It is the framing the rest of the notebook builds on.

### Site selection, briefly and critically

There are two distinct things people mean when they say "site selection":

**Optimization** is the search for the single best site under a fixed set of constraints — the one location that scores highest, the only output that matters. A logistics firm picking a warehouse to minimize delivery time uses optimization. So does a fast-food chain picking a corner with the most foot traffic.

**Shortlisting** is the search for a small set of plausible candidates that a human team will then evaluate in detail. The algorithm narrows. The people decide. **This notebook does shortlisting.** We surface the top fifteen tracts that fit a set of criteria, and we ask you to treat them as starting points for fieldwork, community conversation, and feasibility study — not as answers.

Suitability analysis as a planning method has a long tradition. **Ian McHarg's *Design With Nature* (1969)** showed how to overlay layers of values — slope, soil, viewshed, ecological sensitivity — by hand on tracing paper, producing transparent and legible composite maps. FEMA's flood mapping inherits the same logic. So do school-siting models, hospital-need indices, and the [HUD Affirmatively Furthering Fair Housing](https://www.hud.gov/program_offices/fair_housing_equal_opp/affh) index. Every one of these is a **formalization of someone's values**, and every one looks more objective than it is.

That last sentence is the warning. Be wary of the **authority effect**: a map with a score on it looks more certain than a conversation in a community meeting, but it is not — it is a different kind of argument, with different blind spots. The algorithm can be wrong in a thousand ways the meeting cannot, and the meeting can carry signals the algorithm will never see.

Siting decisions for facilities like cooling centers, shelters, and clinics have historically reflected **political geography as much as need**. Resources go where there is political pressure to put them, which correlates with race, income, and English-language fluency in ways that suitability scores cannot capture. An algorithm that ranks tracts on need will find the tracts with the highest measurable need. It will not find the tracts whose residents have been organizing for twenty years to get a clinic, nor will it find the ones who would benefit most if asked. **Suitability is a starting point. It is not a substitute for politics.**

### Census ACS 5-year (2022) at tract level

The demographic backbone of every scenario in this notebook. We pull income, population, poverty, and the two age-band aggregates we need: **population 65+** and **population under 18**.

The age-band variables are an unfortunate quirk of how the Census publishes ACS: there is no single variable for "people over 65," only a series of one-variable-per-age-band-per-sex columns. We sum them. Same for under-18.

In [ ]:
# Fetch ACS variables with corresponding margin-of-error columns.
ACS_HEADLINE_ESTIMATES = [
    "B19013_001E",  # median household income
    "B01001_001E",  # total population
    "B17001_002E",  # population below poverty
]
# Population 65+ — sum male age-bands B01001_020E..025E and female B01001_044E..049E.
ELDERLY_VARS = [f"B01001_{age:03d}E" for age in [20,21,22,23,24,25,44,45,46,47,48,49]]
# Population under 18 — sum male age-bands B01001_003E..006E and female B01001_027E..030E.
YOUTH_VARS   = [f"B01001_{age:03d}E" for age in [3,4,5,6,27,28,29,30]]

ACS_ESTIMATE_VARIABLES = ACS_HEADLINE_ESTIMATES + ELDERLY_VARS + YOUTH_VARS
ACS_MOE_VARIABLES = [v[:-1] + "M" for v in ACS_ESTIMATE_VARIABLES]

acs_gdf = fetch_census(ACS_ESTIMATE_VARIABLES + ACS_MOE_VARIABLES,
                       geography="tract", year=2022)

# Numeric casting.
for col in ACS_ESTIMATE_VARIABLES + ACS_MOE_VARIABLES:
    if col in acs_gdf.columns:
        acs_gdf[col] = pd.to_numeric(acs_gdf[col], errors="coerce")

# Derived population variables.
acs_gdf["elderly_population"] = acs_gdf[ELDERLY_VARS].sum(axis=1, skipna=True)
acs_gdf["youth_population"]   = acs_gdf[YOUTH_VARS].sum(axis=1, skipna=True)

# Flag tracts where any headline MOE exceeds 30% of the estimate.
def _flag_uncertainty(row):
    """Mark high_uncertainty when any focal MOE / estimate > 0.3."""
    focal = [("B19013_001E", "B19013_001M"),
             ("B01001_001E", "B01001_001M"),
             ("B17001_002E", "B17001_002M")]
    for estimate_col, moe_col in focal:
        estimate = row.get(estimate_col)
        moe = row.get(moe_col)
        if pd.notna(estimate) and pd.notna(moe) and estimate > 0:
            if moe / estimate > 0.3:
                return True
    return False
acs_gdf["high_uncertainty"] = acs_gdf.apply(_flag_uncertainty, axis=1)

print(f"ACS rows: {len(acs_gdf)}")
print(f"High-uncertainty tracts: {acs_gdf['high_uncertainty'].sum()} "
      f"({100*acs_gdf['high_uncertainty'].mean():.1f}%)")
display(acs_gdf[["GEOID", "B01001_001E", "B19013_001E",
                 "elderly_population", "youth_population",
                 "high_uncertainty"]].head(3))

### PLUTO

We fetch a capped sample of NYC's per-lot land-use database. For site selection, the **`ownertype`** field is the single most important variable: `'C'` denotes city-owned property, which is the cheapest and fastest path to siting a public facility. **`bsmtcode`** (the basement code — renamed from `basement` in recent PLUTO releases) gives us a below-grade flag. **`landuse`** lets us filter to publicly compatible uses.

In [ ]:
# PLUTO — sampled. Raise max_rows for full coverage.
PLUTO_COLUMNS = ["bbl", "address", "borough", "lotarea", "bldgarea",
                 "numfloors", "zonedist1", "bldgclass", "ownertype",
                 "landuse", "bsmtcode", "latitude", "longitude"]

pluto_df = fetch_nyc_open_data(
    "64uk-42ks.json",
    params={"$select": ",".join(PLUTO_COLUMNS),
            "$where": "latitude IS NOT NULL AND longitude IS NOT NULL"},
    max_rows=200000,
)

for numeric_col in ["lotarea", "bldgarea", "numfloors", "latitude", "longitude"]:
    if numeric_col in pluto_df.columns:
        pluto_df[numeric_col] = pd.to_numeric(pluto_df[numeric_col], errors="coerce")

# City-owned = ownertype 'C'; below-grade = bsmtcode 1 or 2.
pluto_df["city_owned"]  = pluto_df["ownertype"].astype(str).str.strip() == "C"
pluto_df["below_grade"] = pluto_df["bsmtcode"].astype(str).isin(["1", "2"])

print(f"PLUTO rows: {len(pluto_df)}")
print(f"City-owned share: {100*pluto_df['city_owned'].mean():.1f}%")
print(f"Below-grade share: {100*pluto_df['below_grade'].mean():.1f}%")
display(pluto_df.head(3))

### Street Tree Census 2015

Streets only — parks and private trees are not in this dataset. The 2025–2026 follow-up survey is in progress.

In [ ]:
# Street Tree Census 2015.
TREE_COLUMNS = ["tree_id", "spc_common", "tree_dbh", "status", "health",
                "latitude", "longitude", "nta", "nta_name"]
trees_df = fetch_nyc_open_data(
    "uvpi-gqnh.json",
    params={"$select": ",".join(TREE_COLUMNS),
            "$where": "status = 'Alive'"},
    max_rows=100000,
)
trees_df["tree_dbh"]  = pd.to_numeric(trees_df["tree_dbh"], errors="coerce")
trees_df["latitude"]  = pd.to_numeric(trees_df["latitude"], errors="coerce")
trees_df["longitude"] = pd.to_numeric(trees_df["longitude"], errors="coerce")
print(f"Live street trees: {len(trees_df)}")
display(trees_df.head(3))

### Wi-Fi Hotspots

Public hotspots only. Overcounts in tourist-heavy zones; undercounts home broadband (which is the more meaningful measure of digital access).

In [ ]:
# Wi-Fi Hotspots.
WIFI_COLUMNS = ["provider", "type", "borough", "ntaname",
                "latitude", "longitude"]
wifi_df = fetch_nyc_open_data(
    "yjub-udmw.json",
    params={"$select": ",".join(WIFI_COLUMNS)},
    max_rows=20000,
)
wifi_df["latitude"]  = pd.to_numeric(wifi_df["latitude"], errors="coerce")
wifi_df["longitude"] = pd.to_numeric(wifi_df["longitude"], errors="coerce")
print(f"Wi-Fi hotspots: {len(wifi_df)}")
display(wifi_df.head(3))

### Subway Entrances and ADA (NY State Open Data)

The MTA publishes entrance-level data on **NY State** Open Data, not NYC OD. Entrances carry station IDs; ADA status lives on a companion stations table. We join the two on `station_id`.

In [ ]:
# Subway entrances + ADA join.
entrances_df = fetch_nyc_open_data(
    "i9wp-a4ja.json",
    params={"$select": "station_id,stop_name,division,line,borough,"
                       "entrance_type,entrance_latitude,entrance_longitude"},
    max_rows=5000,
    domain="data.ny.gov",
)
stations_df = fetch_nyc_open_data(
    "39hk-dx4f.json",
    params={"$select": "station_id,ada"},
    max_rows=1000,
    domain="data.ny.gov",
)

if entrances_df.empty:
    print("Subway entrance dataset returned no rows.")
    subway_df = pd.DataFrame()
else:
    entrances_df["latitude"]  = pd.to_numeric(entrances_df["entrance_latitude"],
                                              errors="coerce")
    entrances_df["longitude"] = pd.to_numeric(entrances_df["entrance_longitude"],
                                              errors="coerce")
    subway_df = entrances_df.merge(stations_df, on="station_id", how="left")
    ada_count = (subway_df["ada"].astype(str).str.strip() == "1").sum()
    print(f"Subway entrances: {len(subway_df)} (ADA accessible: {ada_count})")
    display(subway_df.head(3))

### Heat Vulnerability Index

The current DOHMH HVI is published at **ZCTA (ZIP) scale**, not NTA. We pull it as a reference layer and join it to tracts via the ZIP code field in PLUTO. This introduces aggregation error — a tract can contain multiple ZIPs and vice versa — but it gives us a heat-vulnerability signal at tract resolution.

In [ ]:
# Heat Vulnerability Index by ZCTA.
hvi_df = fetch_nyc_open_data("4mhf-duep.json", max_rows=300)
if hvi_df.empty:
    print("HVI dataset returned no rows. Continuing without HVI.")
else:
    hvi_df["zcta20"] = hvi_df["zcta20"].astype(str)
    hvi_df["hvi"]    = pd.to_numeric(hvi_df["hvi"], errors="coerce")
    print(f"HVI rows: {len(hvi_df)}")
    print(f"HVI distribution: min={hvi_df['hvi'].min()} "
          f"max={hvi_df['hvi'].max()} mean={hvi_df['hvi'].mean():.2f}")
    display(hvi_df.head(3))

### 311 Service Requests (last 180 days)

Used to estimate housing and environmental complaint density. Reflects who calls — be careful about reading absence as evidence.

In [ ]:
# 311 calls — recent window, focal complaint types.
from datetime import datetime, timedelta
cutoff = (datetime.utcnow() - timedelta(days=180)).strftime("%Y-%m-%dT00:00:00")
FOCAL_COMPLAINTS = ("HEAT/HOT WATER", "UNSANITARY CONDITION", "NOISE - RESIDENTIAL",
                    "PAINT/PLASTER", "PLUMBING", "RODENT", "AIR QUALITY",
                    "Damaged Tree", "Overgrown Tree/Branches")
complaints_clause = ",".join(f"'{c}'" for c in FOCAL_COMPLAINTS)
calls_df = fetch_nyc_open_data(
    "erm2-nwe9.json",
    params={"$select": "unique_key,created_date,complaint_type,borough,"
                       "latitude,longitude",
            "$where": (f"created_date > '{cutoff}' "
                       f"AND complaint_type IN ({complaints_clause}) "
                       f"AND latitude IS NOT NULL")},
    max_rows=80000,
)
calls_df["latitude"]  = pd.to_numeric(calls_df["latitude"],  errors="coerce")
calls_df["longitude"] = pd.to_numeric(calls_df["longitude"], errors="coerce")
print(f"311 calls: {len(calls_df)}")

### NYC Community Gardens (GreenThumb)

Existing green infrastructure, useful as a complement (gardens already exist and can be expanded) or as a constraint (avoid siting on top of community-managed open space).

**Note:** the canonical Socrata dataset for this is currently marked *archived* by NYC Open Data. The schema we use below is stable, but if you need authoritative current data, cross-reference with the [DPR GreenThumb registry](https://www.greenthumbnyc.org).

In [ ]:
# Community Gardens (archived dataset; still serves point data with lat/lng).
GARDEN_COLUMNS = ["propid", "garden_name", "address", "boro",
                  "size", "jurisdiction", "latitude", "longitude"]
gardens_df = fetch_nyc_open_data(
    "ajxm-kzmj.json",
    params={"$select": ",".join(GARDEN_COLUMNS)},
    max_rows=2000,
)
gardens_df["latitude"]  = pd.to_numeric(gardens_df["latitude"],  errors="coerce")
gardens_df["longitude"] = pd.to_numeric(gardens_df["longitude"], errors="coerce")
print(f"Community gardens: {len(gardens_df)}")
display(gardens_df.head(3))

### Public Library Branches

Filtered from the **NYC Facilities Database**. The Facilities Database is the citywide registry of physical facilities operated or licensed by city agencies; it is the cleanest single source for libraries, health facilities, schools, senior centers, and many other categories. We filter by `facsubgrp = 'PUBLIC LIBRARIES'`.

In [ ]:
# Public Library branches via the Facilities Database.
libraries_df = fetch_nyc_open_data(
    "ji82-xba5.json",
    params={"$select": "facname,address,boro,facsubgrp,latitude,longitude",
            "$where":  "facsubgrp='PUBLIC LIBRARIES'"},
    max_rows=500,
)
libraries_df["latitude"]  = pd.to_numeric(libraries_df["latitude"],  errors="coerce")
libraries_df["longitude"] = pd.to_numeric(libraries_df["longitude"], errors="coerce")
print(f"Public library branches: {len(libraries_df)}")
display(libraries_df.head(3))

### Health Facilities

Filtered from the same Facilities Database. We pull hospitals and clinics, mental health, and other health-care facility types as a broad set.

In [ ]:
# Health facilities via the Facilities Database.
health_df = fetch_nyc_open_data(
    "ji82-xba5.json",
    params={"$select": "facname,address,boro,facsubgrp,latitude,longitude",
            "$where":  "facsubgrp IN ('HOSPITALS AND CLINICS','MENTAL HEALTH',"
                       "'OTHER HEALTH CARE')"},
    max_rows=5000,
)
health_df["latitude"]  = pd.to_numeric(health_df["latitude"],  errors="coerce")
health_df["longitude"] = pd.to_numeric(health_df["longitude"], errors="coerce")
print(f"Health facilities: {len(health_df)} "
      f"({health_df['facsubgrp'].value_counts().to_dict()})")
display(health_df.head(3))

### Floodplain (current 100-year)

NYC's published current FEMA preliminary floodplain is the **"Future Floodplain 2020s"** dataset. The name is awkward — the dataset is named for the decade it describes, and we are in the 2020s — but it is the current published 100-year-floodplain layer at the time of writing.

The geometry comes back as a `the_geom` GeoJSON-style field. We parse it into shapely polygons so we can use it as a polygon constraint.

In [ ]:
# Current 100-year floodplain polygons.
from shapely.geometry import shape
flood_raw = fetch_nyc_open_data(
    "aqw3-vugz.json",
    params=None,
    max_rows=5000,
)
if flood_raw.empty or "the_geom" not in flood_raw.columns:
    print("Floodplain dataset returned no usable geometry.")
    floodplain_gdf = gpd.GeoDataFrame(columns=["fld_zone", "geometry"],
                                      geometry="geometry", crs="EPSG:4326")
else:
    flood_raw["geometry"] = flood_raw["the_geom"].apply(
        lambda raw_geom: shape(raw_geom) if isinstance(raw_geom, dict) else None)
    floodplain_gdf = gpd.GeoDataFrame(
        flood_raw.dropna(subset=["geometry"])[
            [c for c in ["fld_zone", "static_bfe", "geometry"]
             if c in flood_raw.columns]],
        geometry="geometry", crs="EPSG:4326")
    print(f"Floodplain polygons: {len(floodplain_gdf)}")
    if "fld_zone" in floodplain_gdf.columns:
        print(f"Zone codes: {floodplain_gdf['fld_zone'].value_counts().to_dict()}")

### Aggregate point data to census tracts and compute derived variables

Now we bring everything together. For each tract we compute:
- `elderly_pct`, `youth_pct`, `poverty_rate` from ACS
- `hvi_score` — joined from ZCTA-scale HVI by spatially overlaying tract centroids on ZCTA polygons (we approximate by joining via the most common ZIP among PLUTO lots in the tract; **this is the noisiest join in the notebook**)
- `tree_density` — trees per hectare of lot area in the tract
- `transit_access` — subway entrances within 400 m of the tract centroid
- `library_gap` — 1 if no library within 800 m, else 0
- `health_facility_gap` — 1 if no health facility within 800 m, else 0
- `in_flood_zone` — 1 if tract centroid intersects the 100-year floodplain, else 0
- `city_owned_lot_pct` — share of PLUTO lots in the tract that are city-owned
- `wifi_density` — public hotspots per 1,000 residents
- `housing_311_rate` — housing-condition complaints per 1,000 residents

In [ ]:
# Aggregate everything to tract scale.
import numpy as np
from shapely.geometry import Point

def points_from_lat_lng(dataframe):
    """Return a WGS84 GeoDataFrame from a DataFrame with latitude/longitude."""
    valid = dataframe.dropna(subset=["latitude", "longitude"]).copy()
    valid["geometry"] = [Point(lng, lat)
                         for lng, lat in zip(valid["longitude"], valid["latitude"])]
    return gpd.GeoDataFrame(valid, geometry="geometry", crs="EPSG:4326")

trees_gdf      = points_from_lat_lng(trees_df)
wifi_gdf       = points_from_lat_lng(wifi_df)
calls_gdf      = points_from_lat_lng(calls_df)
pluto_gdf      = points_from_lat_lng(pluto_df)
gardens_gdf    = points_from_lat_lng(gardens_df)
libraries_gdf  = points_from_lat_lng(libraries_df)
health_gdf     = points_from_lat_lng(health_df)
subway_gdf     = (points_from_lat_lng(subway_df)
                  if "latitude" in subway_df.columns and not subway_df.empty
                  else gpd.GeoDataFrame(geometry=[], crs="EPSG:4326"))

tracts_gdf = acs_gdf.to_crs(epsg=4326).copy()
tracts_gdf["tract_index"] = range(len(tracts_gdf))

def _sjoin_count(point_gdf, mask=None):
    """Count points falling inside each tract, optionally filtered by mask."""
    if point_gdf.empty:
        return np.zeros(len(tracts_gdf), dtype=int)
    chosen = point_gdf if mask is None else point_gdf[mask]
    if chosen.empty:
        return np.zeros(len(tracts_gdf), dtype=int)
    joined = gpd.sjoin(chosen, tracts_gdf[["tract_index", "geometry"]],
                       how="left", predicate="within")
    counts = joined.groupby("tract_index").size()
    return counts.reindex(tracts_gdf["tract_index"], fill_value=0).values

tracts_gdf["tree_count"]    = _sjoin_count(trees_gdf)
tracts_gdf["wifi_count"]    = _sjoin_count(wifi_gdf)
tracts_gdf["calls_count"]   = _sjoin_count(calls_gdf)
tracts_gdf["pluto_lots"]    = _sjoin_count(pluto_gdf)
tracts_gdf["city_owned_lots"] = _sjoin_count(pluto_gdf,
                                             pluto_gdf["city_owned"] == True)

# Lot area for tree density.
if not pluto_gdf.empty:
    lot_join = gpd.sjoin(pluto_gdf[["lotarea", "geometry"]],
                         tracts_gdf[["tract_index", "geometry"]],
                         how="left", predicate="within")
    lot_area_sqft = lot_join.groupby("tract_index")["lotarea"].sum()
    tracts_gdf["lotarea_ha"] = (lot_area_sqft.reindex(tracts_gdf["tract_index"],
                                                      fill_value=0).values
                                * 0.0000092903)
else:
    tracts_gdf["lotarea_ha"] = 0.0

# Walking-distance access counts (400 m for subway, 800 m for libraries / health).
print("Computing walking-distance access via NY State Plane buffers...")
tracts_state_plane = tracts_gdf.to_crs(epsg=2263).copy()
tracts_state_plane["geometry"] = tracts_state_plane.geometry.centroid

def _count_within_buffer(point_gdf, meters):
    """How many points fall within an n-meter buffer of each tract centroid?"""
    if point_gdf.empty:
        return np.zeros(len(tracts_gdf), dtype=int)
    buffer_ft = meters * 3.28084
    buffered = tracts_state_plane.copy()
    buffered["geometry"] = buffered.geometry.buffer(buffer_ft)
    sp_points = point_gdf.to_crs(epsg=2263)
    joined = gpd.sjoin(sp_points, buffered[["tract_index", "geometry"]],
                       how="left", predicate="within")
    return (joined.groupby("tract_index").size()
            .reindex(tracts_gdf["tract_index"], fill_value=0).values)

tracts_gdf["transit_access"]      = _count_within_buffer(subway_gdf, 400)
tracts_gdf["libraries_within_800m"] = _count_within_buffer(libraries_gdf, 800)
tracts_gdf["health_within_800m"]    = _count_within_buffer(health_gdf, 800)
tracts_gdf["library_gap"]         = (tracts_gdf["libraries_within_800m"] == 0).astype(int)
tracts_gdf["health_facility_gap"] = (tracts_gdf["health_within_800m"] == 0).astype(int)

# Flood-zone overlap (centroid-in-polygon test).
if not floodplain_gdf.empty:
    flood_union = floodplain_gdf.unary_union
    centroids = tracts_gdf.geometry.centroid
    tracts_gdf["in_flood_zone"] = centroids.apply(
        lambda pt: int(flood_union.contains(pt)))
else:
    tracts_gdf["in_flood_zone"] = 0

# Population-normalized rates.
population = pd.to_numeric(tracts_gdf["B01001_001E"], errors="coerce")
tracts_gdf["poverty_rate"]    = np.where(population > 0,
                                         pd.to_numeric(tracts_gdf["B17001_002E"],
                                                       errors="coerce") / population,
                                         np.nan)
tracts_gdf["elderly_pct"]     = np.where(population > 0,
                                         tracts_gdf["elderly_population"] / population,
                                         np.nan)
tracts_gdf["youth_pct"]       = np.where(population > 0,
                                         tracts_gdf["youth_population"] / population,
                                         np.nan)
tracts_gdf["wifi_density"]    = np.where(population > 0,
                                         1000 * tracts_gdf["wifi_count"] / population,
                                         np.nan)
tracts_gdf["housing_311_rate"] = np.where(population > 0,
                                          1000 * tracts_gdf["calls_count"] / population,
                                          np.nan)
tracts_gdf["tree_density"]    = np.where(tracts_gdf["lotarea_ha"] > 0,
                                         tracts_gdf["tree_count"] / tracts_gdf["lotarea_ha"],
                                         0.0)
tracts_gdf["city_owned_lot_pct"] = np.where(tracts_gdf["pluto_lots"] > 0,
                                            100 * tracts_gdf["city_owned_lots"] /
                                            tracts_gdf["pluto_lots"], 0.0)
tracts_gdf["below_grade_pct"] = 0.0  # Re-computed below.
below_join = gpd.sjoin(pluto_gdf[["below_grade", "geometry"]],
                       tracts_gdf[["tract_index", "geometry"]],
                       how="left", predicate="within")
below_share = (below_join.groupby("tract_index")["below_grade"].mean()
               .reindex(tracts_gdf["tract_index"], fill_value=0).values)
tracts_gdf["below_grade_pct"] = below_share * 100.0

# Approximate ZCTA-level HVI -> tract via dominant ZIP among PLUTO lots in the tract.
if not hvi_df.empty and "zipcode" in pluto_gdf.columns:
    pluto_join = gpd.sjoin(pluto_gdf[["zipcode", "geometry"]],
                           tracts_gdf[["tract_index", "geometry"]],
                           how="left", predicate="within")
    dominant_zip = (pluto_join.dropna(subset=["zipcode"])
                    .groupby("tract_index")["zipcode"]
                    .agg(lambda zips: zips.mode().iloc[0]
                         if not zips.mode().empty else None))
    zip_to_hvi = hvi_df.set_index("zcta20")["hvi"].to_dict()
    tracts_gdf["hvi_score"] = dominant_zip.reindex(
        tracts_gdf["tract_index"]).map(zip_to_hvi).values
else:
    tracts_gdf["hvi_score"] = np.nan

tracts_gdf["hvi_score"] = pd.to_numeric(tracts_gdf["hvi_score"], errors="coerce")
print(f"Tracts with HVI score joined: {tracts_gdf['hvi_score'].notna().sum()}"
      f" / {len(tracts_gdf)}")

# Save prepared tract layer.
prepared_path = os.path.join(OUTPUT_FOLDER, "data", "tracts_prepared.geojson")
tracts_gdf.to_file(prepared_path, driver="GeoJSON")
print(f"\nSaved prepared tract layer: {prepared_path}")
print(f"Final tract columns of interest:")
for col in ["elderly_pct","youth_pct","poverty_rate","hvi_score","tree_density",
            "transit_access","library_gap","health_facility_gap","in_flood_zone",
            "city_owned_lot_pct","wifi_density","housing_311_rate","below_grade_pct"]:
    valid = tracts_gdf[col].notna().sum()
    print(f"  {col:<22} non-null: {valid}")

---

## Module 2 — Understanding Suitability Analysis

A primarily educational module. Read carefully — every later scenario builds on this vocabulary.

**The overlay method.** [Ian McHarg's *Design With Nature* (1969)](https://en.wikipedia.org/wiki/Design_with_Nature) introduced systematic suitability analysis to American planning. McHarg drew each value layer — slope, soil drainage, viewshed, ecological sensitivity — on a separate sheet of tracing paper, shading more-suitable areas darker. Stack the layers and the darkest composite areas were the most suitable for development. The method was transparent (you could see every assumption), legible (you could shade in pencil what you cared about), and slow.

**The GIS era.** Suitability analysis moved into GIS in the 1980s, gained computational power, and gradually lost much of its legibility. A modern weighted-overlay can mix forty layers under a black-box raster algebra. The math is the same as McHarg's. The transparency is not.

**The three moving parts of any suitability score.**
1. **Criteria selection.** Which variables matter? This is a values question disguised as a methods question. Every variable you include is a claim about what makes a place suitable; every one you leave out is a claim about what does not.
2. **Normalization.** How do you compare an income (dollars) to a tree count (integers) to a heat-vulnerability score (1–5)? Min-max, z-score, and percentile rank are three common answers — none is correct; each carries different distortions.
3. **Weighting.** How much does each criterion matter relative to the others? A weight is a statement about priority. Doubling a weight doubles the variable's influence on the final ranking.

**Suitability vs. feasibility.** Suitability asks "is this a *good* place for X?" Feasibility asks "can X *actually* be built here?" These are different questions. A high-suitability tract may have no available building, no zoning support, no community willingness, no funding pathway. **Algorithms can ground suitability. They cannot answer feasibility.** Feasibility requires people in rooms.

**The NIMBY problem.** High-suitability sites often face the most concentrated community opposition — exactly because the suitability score points to places where the need is greatest and the politics are hardest. Cooling centers, shelters, supportive housing, and methadone clinics are the canonical examples. **An algorithm cannot model political feasibility.** It cannot tell you whether a community board will fight a proposal, or whether a council member will be voted out for supporting it.

### Worked example: how a suitability score is computed

Below we walk a tiny five-row toy dataset through the three components. Read the printed dataframe at each step — they show the transformation in slow motion. Once you see what is happening here, every later scenario in the notebook is just this same arithmetic at city scale.

In [ ]:
# Worked example — what compute_suitability is doing under the hood.
import pandas as pd

toy = pd.DataFrame({
    "tract_id":           ["A", "B", "C", "D", "E"],
    "hvi_score":          [4, 2, 5, 1, 3],          # 1-5, higher = worse heat
    "city_owned_lot_pct": [5.0, 30.0, 12.0, 0.0, 18.0],
    "in_flood_zone":      [0, 0, 1, 0, 1],          # boolean
})
print("Step 0 — raw toy data:")
print(toy)
print()

# Step 1 — min-max normalize each column to [0, 1].
normalized = toy.copy()
for col in ["hvi_score", "city_owned_lot_pct", "in_flood_zone"]:
    mn = toy[col].min(); mx = toy[col].max()
    normalized[col + "_norm"] = (toy[col] - mn) / (mx - mn) if mx != mn else 0.0
print("Step 1 — min-max normalization (each column rescaled to [0, 1]):")
print(normalized[["tract_id", "hvi_score_norm", "city_owned_lot_pct_norm",
                  "in_flood_zone_norm"]])
print()

# Step 2 — apply direction (flood zone is bad, so invert).
directed = normalized.copy()
directed["in_flood_zone_dir"] = 1.0 - normalized["in_flood_zone_norm"]
directed["hvi_score_dir"]     = normalized["hvi_score_norm"]
directed["city_owned_lot_pct_dir"] = normalized["city_owned_lot_pct_norm"]
print("Step 2 — direction-corrected (in_flood_zone inverted because lower = better):")
print(directed[["tract_id","hvi_score_dir","city_owned_lot_pct_dir","in_flood_zone_dir"]])
print()

# Step 3 — apply weights and sum.
WEIGHTS = {"hvi_score_dir": 0.5,
           "city_owned_lot_pct_dir": 0.3,
           "in_flood_zone_dir": 0.2}
directed["suitability_score"] = sum(directed[col] * w for col, w in WEIGHTS.items())
print("Step 3 — weighted sum (weights sum to 1.0):")
print(directed[["tract_id", "suitability_score"]].round(3))
print()
print("Read top to bottom: which tract scored highest, and why?")

---

## Module 3 — Example Site Selections

Three worked scenarios. Each one is a complete argument: a use, a set of criteria, a weighting scheme, and a shortlist. Read them as examples of how the same suitability machinery answers very different questions.

**Before any scenario:** every criterion is a claim about what makes a site suitable, and every weight is a claim about how much that criterion matters relative to the others. **If you weight transit access highly, you are designing for people who already have transit access.** Every weighting decision is an answer to "who is this facility for?" Make those choices explicit and defensible.

**Module 3 depends on:** Modules 0 and 1.

In [ ]:
# Shared scoring + display infrastructure.
import os
import folium
import matplotlib.pyplot as plt
import geopandas as gpd

# Load the prepared tract layer from Module 1 so this module is independently runnable.
tracts_gdf = gpd.read_file(os.path.join(OUTPUT_FOLDER, "data",
                                        "tracts_prepared.geojson")).to_crs(epsg=4326)
floodplain_gdf = (gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
                  if "floodplain_gdf" not in globals() else floodplain_gdf)

def render_scenario(scored_gdf, criteria, scenario_title, map_filename,
                    facility_layer=None, facility_layer_name="Existing facilities",
                    n=None):
    """Render a folium map + top/bottom bar chart and save outputs."""
    top_n = n if n is not None else TOP_N
    valid = scored_gdf.dropna(subset=["suitability_score"]).copy()
    ranked = valid.sort_values("suitability_score", ascending=False)
    top_rows    = ranked.head(top_n)
    bottom_rows = ranked.tail(top_n).iloc[::-1]

    # Folium choropleth.
    centroids_lat = valid.geometry.centroid.y.mean()
    centroids_lng = valid.geometry.centroid.x.mean()
    out_map = folium.Map(location=[centroids_lat, centroids_lng],
                         zoom_start=11, tiles="cartodbpositron")
    folium.Choropleth(
        geo_data=valid.__geo_interface__,
        data=valid,
        columns=["GEOID", "suitability_score"],
        key_on="feature.properties.GEOID",
        fill_color="YlGnBu", fill_opacity=0.75, line_opacity=0.15,
        legend_name=scenario_title,
    ).add_to(out_map)
    folium.GeoJson(
        valid,
        style_function=lambda feature: {"fillOpacity": 0,
                                        "color": "#333", "weight": 0.3},
        tooltip=folium.features.GeoJsonTooltip(
            fields=["GEOID", "suitability_score"] + [c["variable"] for c in criteria],
            aliases=["Tract", "Score"] + [c["variable"] for c in criteria],
            localize=True),
        name="Tracts",
    ).add_to(out_map)
    # Top-N highlight.
    folium.GeoJson(
        top_rows,
        style_function=lambda feature: {"fillOpacity": 0,
                                        "color": "#3C4ED6", "weight": 3},
        name=f"Top {top_n} candidates",
    ).add_to(out_map)
    # Existing facilities (if relevant for the scenario).
    if facility_layer is not None and not facility_layer.empty:
        facility_group = folium.FeatureGroup(name=facility_layer_name, show=True)
        for _, row in facility_layer.dropna(subset=["latitude","longitude"]).iterrows():
            folium.CircleMarker(
                location=[float(row["latitude"]), float(row["longitude"])],
                radius=3, color="#7B241C", fill=True, fill_opacity=0.85,
                tooltip=row.get("facname", row.get("garden_name", ""))
            ).add_to(facility_group)
        facility_group.add_to(out_map)
    # FEMA flood zone toggle.
    if not floodplain_gdf.empty:
        folium.GeoJson(
            floodplain_gdf,
            style_function=lambda feature: {"fillColor": "#3C4ED6",
                                            "color": "#3C4ED6",
                                            "fillOpacity": 0.25, "weight": 0.5},
            name="100-year floodplain",
            show=False,
        ).add_to(out_map)

    folium.LayerControl(collapsed=False).add_to(out_map)
    map_path = os.path.join(OUTPUT_FOLDER, "maps", map_filename)
    out_map.save(map_path)
    print(f"Saved map: {map_path}")

    # Bar chart.
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    axes[0].barh(top_rows["GEOID"], top_rows["suitability_score"], color="#3C4ED6")
    axes[0].invert_yaxis(); axes[0].set_title(f"Top {top_n} — {scenario_title}")
    axes[1].barh(bottom_rows["GEOID"], bottom_rows["suitability_score"], color="#C0392B")
    axes[1].invert_yaxis(); axes[1].set_title(f"Bottom {top_n} — {scenario_title}")
    plt.tight_layout(); plt.show()

    # Export top-N.
    export_path = os.path.join(OUTPUT_FOLDER, "exports",
                               map_filename.replace(".html", "_topN.geojson"))
    top_rows.to_file(export_path, driver="GeoJSON")
    print(f"Saved top-{top_n} candidates: {export_path}")

    return out_map, top_rows, bottom_rows

print("Scenario infrastructure ready. tracts_gdf has",
      len(tracts_gdf), "tracts.")

### Scenario A — Cooling Center Siting

**What we are siting.** A publicly accessible cooling center — a place that residents can walk to on the hottest days of the year, equipped with air conditioning, drinking water, and seating.

**Who it serves.** The model targets tracts where (a) heat vulnerability is highest, (b) the populations most at risk for heat-related mortality (elderly, low-income) live in highest concentration, and (c) public transit makes the center reachable without a car. The model also prefers tracts with city-owned land available, and penalizes tracts in the floodplain.

| Criterion | Weight | Direction | Rationale |
|---|---|---|---|
| `hvi_score` | 0.30 | Higher = more suitable | Primary need driver |
| `elderly_pct` | 0.20 | Higher = more suitable | Highest heat-mortality risk |
| `poverty_rate` | 0.15 | Higher = more suitable | Lower likelihood of home AC |
| `transit_access` | 0.15 | Higher = more suitable | Must be reachable without a car |
| `city_owned_lot_pct` | 0.10 | Higher = more suitable | Public land lowers acquisition cost |
| `in_flood_zone` | 0.10 | **Inverted** | Must operate in climate emergencies |

**Normalization: percentile rank.** Robust to outliers. Our HVI signal in particular has compressed range (1–5), so percentile is more legible than min-max.

**What this model does not see.** The size of available city-owned lots. The condition of existing buildings. Interior square footage. ADA accessibility of structures actually present. A tract may score in the top fifteen and have nothing physically siteable in it. Suitability is not feasibility.

In [ ]:
# Scenario A — Cooling Centers.
cooling_criteria = [
    {"variable": "hvi_score",          "weight": 0.30, "invert": False},
    {"variable": "elderly_pct",        "weight": 0.20, "invert": False},
    {"variable": "poverty_rate",       "weight": 0.15, "invert": False},
    {"variable": "transit_access",     "weight": 0.15, "invert": False},
    {"variable": "city_owned_lot_pct", "weight": 0.10, "invert": False},
    {"variable": "in_flood_zone",      "weight": 0.10, "invert": True},
]

# Print the criteria table.
criteria_table = pd.DataFrame(cooling_criteria)
print("Scenario A criteria:"); display(criteria_table)

scored_a = compute_suitability(tracts_gdf, cooling_criteria, normalizer="percentile")
render_scenario(scored_a, cooling_criteria,
                "Cooling Center Suitability", "scenario_a_cooling_centers.html",
                facility_layer=None)

**Winners and losers — Scenario A.**

- These are the top fifteen candidate tracts. What do they have in common spatially? Are they clustered in certain neighborhoods?
- What tracts scored lowest, and what does that mean for the populations there? A low score does not mean those tracts do not have heat-vulnerable residents — it means the *combination* of vulnerability, demographics, and feasibility puts other tracts higher.
- Is there a tract you know should be on this list that is not? What criterion or data gap might explain its absence?

### Scenario B — Street Tree Planting Priority Areas

**What we are siting.** Not a building — a planting effort. Identify the tracts where new street tree planting would deliver the greatest equity benefit.

**Who it serves.** The model targets tracts with the highest heat vulnerability, the fewest existing trees, the highest poverty rate, the most heat-vulnerable populations, and an indirect built-environment density signal.

| Criterion | Weight | Direction | Rationale |
|---|---|---|---|
| `hvi_score` | 0.25 | Higher = more suitable | Highest cooling benefit |
| `tree_density` | 0.25 | **Inverted** | Areas with fewest trees need the most planting |
| `poverty_rate` | 0.20 | Higher = more suitable | Green infrastructure gaps concentrate in low-income areas |
| `below_grade_pct` (impervious proxy) | 0.15 | Higher = more suitable | Dense built environment with less soil availability |
| `elderly_pct` | 0.15 | Higher = more suitable | Shade benefit greatest for heat-vulnerable populations |

**Normalization: min-max.** Tree density has a long tail (a few tracts with park frontage have outsized counts); min-max lets that outlier signal through. We trade off interpretability for sensitivity.

**A weak proxy worth naming.** `below_grade_pct` is being used here as a stand-in for impervious surface and built-environment density. The relationship is real but indirect — a tract with many below-grade buildings tends to be older and denser. **A direct impervious-surface measurement from satellite imagery or NYC's tree canopy mosaic would be more accurate.** We use what is in open data; the proxy is documented.

**What this model does not see.** Sidewalk width. Underground utility conflicts. Soil volume for tree pits. Species-specific suitability. **This identifies need, not plantability.**

In [ ]:
# Scenario B — Tree Planting Priority Areas.
tree_criteria = [
    {"variable": "hvi_score",        "weight": 0.25, "invert": False},
    {"variable": "tree_density",     "weight": 0.25, "invert": True},
    {"variable": "poverty_rate",     "weight": 0.20, "invert": False},
    {"variable": "below_grade_pct",  "weight": 0.15, "invert": False},
    {"variable": "elderly_pct",      "weight": 0.15, "invert": False},
]
print("Scenario B criteria:")
display(pd.DataFrame(tree_criteria))

scored_b = compute_suitability(tracts_gdf, tree_criteria, normalizer="min_max")
render_scenario(scored_b, tree_criteria,
                "Tree Planting Priority", "scenario_b_tree_planting.html",
                facility_layer=None)

**Winners and losers — Scenario B.**

- The top-fifteen tracts — what do they look like as a spatial pattern? Do they trace neighborhood lines, borough lines, or something more granular?
- The bottom tracts are not "tree-rich" — they may simply already be at tree saturation, or they may have demographics where the equity weighting did not surface them. Which is it for your bottom-fifteen?
- Whose lived experience of urban heat is not in this model? Outdoor workers, unhoused residents, people who work daytime shifts in air-conditioned offices but live in heat-vulnerable apartments — the model sees none of them directly.

### Scenario C — Public Library Branch Gap Analysis

**What we are siting.** A new library branch (or programming partnership). Identify tracts that are underserved by the existing library system, weighted by the populations that disproportionately benefit from libraries.

**Who it serves.** Tracts with no library within 800 m, high poverty, high share of young people (libraries' core constituency), strong transit access (for a library to function as a neighborhood anchor people can reach), and low public wi-fi density (where library internet would close a measurable gap).

| Criterion | Weight | Direction | Rationale |
|---|---|---|---|
| `library_gap` | 0.35 | Higher = more suitable | The primary need signal — no nearby branch |
| `poverty_rate` | 0.20 | Higher = more suitable | Libraries' equity mission |
| `youth_pct` | 0.20 | Higher = more suitable | Libraries disproportionately serve youth |
| `transit_access` | 0.15 | Higher = more suitable | Reachable from beyond the block |
| `wifi_density` | 0.10 | **Inverted** | Less public broadband = greater library internet need |

**Normalization: z-score.** The variables here have very different scales (a 0/1 gap flag vs. percentages vs. integer counts). Z-score puts each on a common standard-deviation footing.

**What this model does not see.** Branch size, hours, language services, collection relevance to the population. A tract may be "served" by a branch that is open ten hours a week, has no books in the dominant language of the neighborhood, and lacks programming for the age group most present. **Proximity is not access.**

In [ ]:
# Scenario C — Library Gap Analysis.
library_criteria = [
    {"variable": "library_gap",   "weight": 0.35, "invert": False},
    {"variable": "poverty_rate",  "weight": 0.20, "invert": False},
    {"variable": "youth_pct",     "weight": 0.20, "invert": False},
    {"variable": "transit_access","weight": 0.15, "invert": False},
    {"variable": "wifi_density",  "weight": 0.10, "invert": True},
]
print("Scenario C criteria:")
display(pd.DataFrame(library_criteria))

scored_c = compute_suitability(tracts_gdf, library_criteria, normalizer="zscore")
render_scenario(scored_c, library_criteria,
                "Library Branch Gap", "scenario_c_library_gap.html",
                facility_layer=libraries_gdf, facility_layer_name="Existing libraries")

**Winners and losers — Scenario C.**

- Top fifteen tracts on the library-gap map — overlay them mentally against the existing library dots. Is the algorithm identifying the kinds of gaps you would have identified by eye, or different ones?
- The lowest-scoring tracts are not "library-saturated." Some of them are wealthy enough that residents have their own internet, their own books, their own access to information services that bypass the public library entirely. The model treats them as well-served because the gap variable is zero. Is that the same as well-served?
- The most striking absence from this model: the populations who depend on libraries most heavily — unhoused people who use libraries for shelter and computer access, recent immigrants who use them for English classes and document help — are not measurable from the open-data variables here. The model surfaces *demographic correlates* of library demand, not library demand itself.

---

## Module 4 — Define Your Own Site Selection

You have seen three scenarios. Each made a claim about what matters for a particular use. Now you make that claim yourself.

**Start with the use.** What are you siting, and who does it serve? Let that answer drive your criteria. Do not start with the data and work backward to a use — that produces models that score what is measurable, not what matters.

**Module 4 depends on:** Modules 0, 1, and (optionally) 2 / 3 for context.

In [ ]:
# Module 4 — interactive site selection builder.
import os
import datetime
import folium
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Reload prepared tracts so this module runs independently.
tracts_gdf = gpd.read_file(os.path.join(OUTPUT_FOLDER, "data",
                                        "tracts_prepared.geojson")).to_crs(epsg=4326)

# Variable catalog with one-line descriptions.
VARIABLE_CATALOG = {
    "hvi_score":           "Heat Vulnerability Index (1–5, higher = more vulnerable)",
    "elderly_pct":         "Share of population aged 65+",
    "youth_pct":           "Share of population under 18",
    "poverty_rate":        "Share of population below poverty line",
    "tree_density":        "Street trees per hectare of lot area",
    "transit_access":      "Subway entrances within 400 m of tract centroid",
    "library_gap":         "1 if no library within 800 m, else 0",
    "health_facility_gap": "1 if no health facility within 800 m, else 0",
    "in_flood_zone":       "1 if tract centroid is in 100-year floodplain",
    "city_owned_lot_pct":  "Share of PLUTO lots that are city-owned",
    "wifi_density":        "Public wi-fi hotspots per 1,000 residents",
    "housing_311_rate":    "Housing-condition 311 calls per 1,000 residents",
    "below_grade_pct":     "Share of PLUTO lots with below-grade space",
}

NORMALIZER_EXPLANATIONS = {
    "min_max":    "Each value rescaled to [0,1] using min/max. Sensitive to outliers.",
    "percentile": "Each tract ranked against every other tract. Robust to outliers.",
    "zscore":     "Each value centered on mean, scaled by std. Good for surfacing extremes.",
}

# Step 1 — use definition.
use_input = widgets.Text(value="", placeholder="e.g., a youth mental health clinic",
                         description="What are you siting?",
                         style={"description_width": "initial"},
                         layout=widgets.Layout(width="700px"))
population_input = widgets.Text(value="", placeholder="e.g., low-income teenagers in transit-poor areas",
                                description="Who does it serve?",
                                style={"description_width": "initial"},
                                layout=widgets.Layout(width="700px"))

# Step 2 — hard constraints.
constraint_flood    = widgets.Checkbox(value=True, description="Exclude tracts in 100-year floodplain")
constraint_city_lot = widgets.Checkbox(value=False, description="Include only tracts with ≥1 city-owned lot")
constraint_pop_floor = widgets.Checkbox(value=False, description="Exclude tracts with population < 1,000")

# Step 3 — variable selection.
variable_picker = widgets.SelectMultiple(
    options=[(f"{name} — {desc}", name) for name, desc in VARIABLE_CATALOG.items()],
    value=("hvi_score", "poverty_rate", "transit_access"),
    rows=12, description="Variables:",
    layout=widgets.Layout(width="800px"),
    style={"description_width": "initial"})

# Steps 4–6 — per-variable controls, rebuilt dynamically.
direction_box   = widgets.VBox([])
weight_box      = widgets.VBox([])
rationale_box   = widgets.VBox([])
weight_total_label = widgets.HTML(value="<b>Total weight: 0.00</b>")

direction_toggles = {}
weight_sliders    = {}
rationale_inputs  = {}

normalization_picker = widgets.Dropdown(
    options=[("Min-Max","min_max"), ("Percentile Rank","percentile"), ("Z-Score","zscore")],
    value="min_max", description="Normalization:",
    style={"description_width": "initial"})
normalization_help = widgets.HTML(
    value=f"<i>{NORMALIZER_EXPLANATIONS['min_max']}</i>")

def _on_norm_change(change):
    normalization_help.value = f"<i>{NORMALIZER_EXPLANATIONS[change['new']]}</i>"
normalization_picker.observe(_on_norm_change, names="value")

limitations_input = widgets.Textarea(
    value="", placeholder="What does this model miss?",
    description="Limitations:",
    layout=widgets.Layout(width="700px", height="80px"),
    style={"description_width": "initial"})
missing_data_input = widgets.Textarea(
    value="", placeholder="What data did you wish you had but could not include?",
    description="Data I wish I had:",
    layout=widgets.Layout(width="700px", height="80px"),
    style={"description_width": "initial"})

def _on_weight_change(_):
    total = sum(slider.value for slider in weight_sliders.values())
    color = "#117864" if abs(total - 1.0) <= 0.01 else "#C0392B"
    label = "OK" if abs(total - 1.0) <= 0.01 else "must equal 1.00"
    weight_total_label.value = (f"<b style='color:{color}'>Total weight: "
                                f"{total:.2f} — {label}</b>")

def _rebuild_variable_controls(*_):
    selected = list(variable_picker.value)
    direction_toggles.clear(); weight_sliders.clear(); rationale_inputs.clear()
    direction_box.children = []; weight_box.children = []; rationale_box.children = []
    if not selected:
        weight_total_label.value = "<b>Total weight: 0.00</b>"
        return
    default_weight = round(1.0 / len(selected), 2)
    direction_children, weight_children, rationale_children = [], [], []
    for variable_name in selected:
        toggle = widgets.ToggleButtons(
            options=[("Higher = More suitable","higher_better"),
                     ("Higher = Less suitable","higher_worse")],
            value="higher_better", description=variable_name,
            style={"description_width": "initial"})
        direction_toggles[variable_name] = toggle
        direction_children.append(toggle)
        slider = widgets.FloatSlider(
            value=default_weight, min=0.0, max=1.0, step=0.05,
            description=variable_name,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="650px"))
        weight_sliders[variable_name] = slider
        slider.observe(_on_weight_change, names="value")
        weight_children.append(slider)
        rationale = widgets.Text(
            value="", placeholder=f"Why is {variable_name} in this model?",
            description=variable_name,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="750px"))
        rationale_inputs[variable_name] = rationale
        rationale_children.append(rationale)
    direction_box.children = direction_children
    weight_box.children    = weight_children
    rationale_box.children = rationale_children
    _on_weight_change(None)
variable_picker.observe(_rebuild_variable_controls, names="value")
_rebuild_variable_controls()

run_button = widgets.Button(description="Run site selection",
                            button_style="primary", icon="play")
output_panel = widgets.Output()

def _format_methodology_card(payload):
    """Return a formatted plain-text site-selection methodology card."""
    lines = ["=" * 76,
             f"SITE SELECTION METHODOLOGY CARD",
             "=" * 76,
             f"Use                       : {payload['use']}",
             f"Primary population served : {payload['population']}",
             f"Hard constraints applied  : {', '.join(payload['constraints']) or '(none)'}",
             f"Geography                 : Census Tract",
             f"Normalization method      : {payload['normalization']}",
             f"  -> {payload['normalization_explanation']}",
             f"Tracts after constraints  : {payload['tracts_after_filter']} "
             f"of {payload['tracts_before_filter']}",
             f"Date generated            : {payload['generated_at']}",
             "",
             "Criteria, weights, directions:"]
    for criterion in payload["criteria"]:
        direction_label = ("Higher = More suitable" if not criterion["invert"]
                           else "Higher = Less suitable")
        lines.append(f"  - {criterion['variable']:<22} "
                     f"weight={criterion['weight']:.2f}  {direction_label}")
        if criterion.get("rationale"):
            lines.append(f"      rationale: {criterion['rationale']}")
    lines.append("")
    lines.append("Data I wish I had:")
    lines.append(f"  {payload['missing_data'] or '(left blank by the author)'}")
    lines.append("")
    lines.append("Known limitations:")
    lines.append(f"  {payload['limitations'] or '(left blank by the author)'}")
    lines.append("")
    lines.append(f"Top {payload['top_n']} candidate tracts:")
    for tract in payload["top_candidates"]:
        lines.append(f"  {tract['rank']:>2}. {tract['geoid']:<13} "
                     f"score={tract['score']:.3f}")
    lines.append("=" * 76)
    return "\n".join(lines)

def _save_card_pdf(card_text, pdf_path):
    """Render the card text onto a single matplotlib page and save as PDF."""
    fig = plt.figure(figsize=(8.5, 11))
    fig.text(0.05, 0.97, card_text, family="monospace",
             fontsize=8, va="top", ha="left")
    plt.axis("off")
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
    plt.close(fig)

def _apply_hard_constraints(gdf):
    """Return tracts after applying hard constraints + names of applied filters."""
    filtered = gdf.copy()
    applied = []
    if constraint_flood.value and "in_flood_zone" in filtered.columns:
        filtered = filtered[filtered["in_flood_zone"] != 1]
        applied.append("Exclude floodplain")
    if constraint_city_lot.value and "city_owned_lots" in filtered.columns:
        filtered = filtered[filtered["city_owned_lots"] >= 1]
        applied.append("Require city-owned lot")
    if constraint_pop_floor.value and "B01001_001E" in filtered.columns:
        filtered = filtered[pd.to_numeric(filtered["B01001_001E"],
                                          errors="coerce").fillna(0) >= 1000]
        applied.append("Population >= 1,000")
    return filtered, applied

def _on_run_clicked(_):
    with output_panel:
        output_panel.clear_output()
        selected = list(variable_picker.value)
        if len(selected) < 3 or len(selected) > 8:
            print(f"Select between 3 and 8 variables (you selected {len(selected)}).")
            return
        total_weight = sum(s.value for s in weight_sliders.values())
        if abs(total_weight - 1.0) > 0.01:
            print(f"Weights must sum to 1.0. Current total: {total_weight:.2f}.")
            return

        criteria = []
        for variable_name in selected:
            criteria.append({
                "variable":  variable_name,
                "weight":    float(weight_sliders[variable_name].value),
                "invert":    direction_toggles[variable_name].value == "higher_worse",
                "rationale": rationale_inputs[variable_name].value.strip(),
            })

        before_count = len(tracts_gdf)
        filtered_gdf, applied_constraints = _apply_hard_constraints(tracts_gdf)
        if len(filtered_gdf) == 0:
            print("No tracts remain after applying hard constraints. "
                  "Loosen at least one.")
            return
        print(f"Applied constraints: {applied_constraints or 'none'}")
        print(f"Tracts before constraints: {before_count}")
        print(f"Tracts after constraints : {len(filtered_gdf)}")

        scored = compute_suitability(filtered_gdf, criteria,
                                     normalizer=normalization_picker.value)
        ranked = scored.dropna(subset=["suitability_score"]).sort_values(
            "suitability_score", ascending=False)
        top_rows    = ranked.head(TOP_N)
        bottom_rows = ranked.tail(TOP_N).iloc[::-1]

        payload = {
            "use":            use_input.value.strip() or "(no use defined)",
            "population":     population_input.value.strip() or "(no population defined)",
            "constraints":    applied_constraints,
            "normalization":  normalization_picker.value,
            "normalization_explanation": NORMALIZER_EXPLANATIONS[normalization_picker.value],
            "generated_at":   datetime.datetime.utcnow().isoformat(timespec="seconds")+"Z",
            "criteria":       criteria,
            "limitations":    limitations_input.value.strip(),
            "missing_data":   missing_data_input.value.strip(),
            "tracts_before_filter": before_count,
            "tracts_after_filter":  len(filtered_gdf),
            "top_n":          TOP_N,
            "top_candidates": [{"rank": i+1,
                                "geoid": str(getattr(row, "GEOID", "?")),
                                "score": float(row.suitability_score)}
                               for i, row in enumerate(top_rows.itertuples())],
        }

        card_text = _format_methodology_card(payload)
        print(card_text)

        slug = "".join(c if c.isalnum() else "_" for c in payload["use"])[:50] or "site_selection"
        timestamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
        txt_path = os.path.join(OUTPUT_FOLDER, "cards", f"{slug}_{timestamp}.txt")
        pdf_path = os.path.join(OUTPUT_FOLDER, "cards", f"{slug}_{timestamp}.pdf")
        with open(txt_path, "w") as text_file:
            text_file.write(card_text)
        _save_card_pdf(card_text, pdf_path)
        print(f"\nSaved methodology card (text): {txt_path}")
        print(f"Saved methodology card (PDF) : {pdf_path}")

        # Map + bars.
        center_lat = scored.geometry.centroid.y.mean()
        center_lng = scored.geometry.centroid.x.mean()
        student_map = folium.Map(location=[center_lat, center_lng],
                                 zoom_start=11, tiles="cartodbpositron")
        folium.Choropleth(
            geo_data=scored.__geo_interface__,
            data=scored, columns=["GEOID", "suitability_score"],
            key_on="feature.properties.GEOID",
            fill_color="YlGnBu", fill_opacity=0.78, line_opacity=0.15,
            legend_name=payload["use"],
        ).add_to(student_map)
        folium.GeoJson(
            top_rows,
            style_function=lambda feature: {"fillOpacity": 0,
                                            "color": "#3C4ED6", "weight": 3},
            name=f"Top {TOP_N} candidates",
        ).add_to(student_map)
        folium.LayerControl().add_to(student_map)
        display(student_map)

        fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
        axes[0].barh(top_rows["GEOID"], top_rows["suitability_score"], color="#3C4ED6")
        axes[0].invert_yaxis(); axes[0].set_title(f"Top {TOP_N}")
        axes[1].barh(bottom_rows["GEOID"], bottom_rows["suitability_score"], color="#C0392B")
        axes[1].invert_yaxis(); axes[1].set_title(f"Bottom {TOP_N}")
        plt.tight_layout(); plt.show()

        # Exports.
        scored.to_file(os.path.join(OUTPUT_FOLDER, "exports",
                                    "site_selection.geojson"), driver="GeoJSON")
        scored.drop(columns="geometry").to_csv(
            os.path.join(OUTPUT_FOLDER, "exports", "site_selection.csv"), index=False)
        with open(os.path.join(OUTPUT_FOLDER, "exports",
                               "site_selection_payload.json"), "w") as payload_file:
            json.dump(payload, payload_file, indent=2)
        print("\nExports written to exports/site_selection.{geojson,csv,payload.json}")

run_button.on_click(_on_run_clicked)

display(widgets.VBox([
    widgets.HTML("<h4>Step 1 — Define the use</h4>"),
    use_input, population_input,
    widgets.HTML("<h4>Step 2 — Hard constraints (eliminate tracts before scoring)</h4>"
                 "<i>Constraints are disqualifying. A constraint is not a low weight — "
                 "it removes tracts from consideration entirely.</i>"),
    constraint_flood, constraint_city_lot, constraint_pop_floor,
    widgets.HTML("<h4>Step 3 — Criterion selection (3 to 8 variables)</h4>"),
    variable_picker,
    widgets.HTML("<h4>Step 4 — Direction</h4>"),
    direction_box,
    widgets.HTML("<h4>Step 5 — Normalization</h4>"),
    normalization_picker, normalization_help,
    widgets.HTML("<h4>Step 6 — Weighting (must sum to 1.0)</h4>"),
    weight_box, weight_total_label,
    widgets.HTML("<h4>Per-criterion rationale (one sentence each)</h4>"),
    rationale_box,
    widgets.HTML("<h4>Reflection inputs (filled into the methodology card)</h4>"),
    missing_data_input, limitations_input,
    widgets.HTML("<h4>Step 7 — Run</h4>"),
    run_button, output_panel,
]))

import json  # used by _on_run_clicked payload dump

**Winners and losers — your scenario.**

Look at your top `TOP_N` tracts on the map. Are they clustered? Do they share characteristics beyond your scoring criteria — borough, density, proximity to existing infrastructure?

Now look at the bottom. **What would it mean for those communities if this facility were never built near them?** A low score in your model is not an absence of need. It is an absence of fit between the *measurable* need you defined and the data we have. People in those tracts still have heat, still need libraries, still need health care.

**The missing data problem.**

You listed data you wished you had. For each item:

- **Who would need to collect it?** A city agency, a community organization, an academic team, the residents themselves?
- **How would they collect it?** Survey, sensors, administrative records, ethnographic fieldwork?
- **Whose cooperation would be required?** Building owners, landlords, language interpreters, people willing to talk to the city?

**What does the absence of that data tell you about whose experience is legible to city government and whose is not?** The data we have is the data the city already collects, which is the data the city already values. The gap between what is measured and what matters is where good site-selection work begins.

---

## Module 5 — Export and Final Map

A final layered map that brings your scenario into a single view. Toggle layers on and off to read the candidate set against the existing facility landscape, the flood zones, and the 311 complaint geography.

**Module 5 depends on:** Module 4 (we read `exports/site_selection.geojson`).

In [ ]:
# Module 5 — final layered map.
import os
import folium
import pandas as pd
import geopandas as gpd
from folium.plugins import HeatMap

student_path = os.path.join(OUTPUT_FOLDER, "exports", "site_selection.geojson")
if not os.path.exists(student_path):
    print("No student site selection found. Run Module 4 first.")
    raise SystemExit

scored = gpd.read_file(student_path).to_crs(epsg=4326)
top_rows = (scored.dropna(subset=["suitability_score"])
            .sort_values("suitability_score", ascending=False).head(TOP_N))

# Pull point data needed for facility / 311 layers.
tracts_for_center = scored.copy()
center = [tracts_for_center.geometry.centroid.y.mean(),
          tracts_for_center.geometry.centroid.x.mean()]
final_map = folium.Map(location=center, zoom_start=11, tiles="cartodbpositron")

# Base: tract suitability choropleth.
folium.Choropleth(
    geo_data=scored.__geo_interface__,
    data=scored, columns=["GEOID", "suitability_score"],
    key_on="feature.properties.GEOID",
    fill_color="YlGnBu", fill_opacity=0.78, line_opacity=0.15,
    legend_name="Suitability score",
).add_to(final_map)

folium.GeoJson(
    scored,
    style_function=lambda feature: {"fillOpacity": 0,
                                    "color": "#333", "weight": 0.3},
    tooltip=folium.features.GeoJsonTooltip(
        fields=["GEOID", "suitability_score"],
        aliases=["Tract", "Score"], localize=True),
    name="Tracts",
).add_to(final_map)

# Top-N: bold border and a numbered label.
for rank, (_, row) in enumerate(top_rows.iterrows(), start=1):
    folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda feature: {"fillOpacity": 0,
                                        "color": "#3C4ED6", "weight": 3},
        tooltip=f"#{rank} — {row.GEOID} (score {row.suitability_score:.3f})",
    ).add_to(final_map)
    centroid = row.geometry.centroid
    folium.map.Marker(
        [centroid.y, centroid.x],
        icon=folium.DivIcon(
            html=f'<div style="font-size:11px;color:#1B1B33;background:white;'
                 f'border:1px solid #3C4ED6;border-radius:50%;width:22px;'
                 f'height:22px;text-align:center;line-height:22px;font-weight:bold;">'
                 f'{rank}</div>')
    ).add_to(final_map)

# Existing facility layers (loaded from cached CSVs if available).
def _try_csv(path):
    """Return DataFrame from CSV if it exists, else an empty DataFrame."""
    return pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()

# We re-pull facility data here rather than depending on in-memory state from
# Module 1, so this cell is independently runnable after a runtime reset.
print("Re-fetching facility points for the final map...")
libraries_df = fetch_nyc_open_data(
    "ji82-xba5.json",
    params={"$select": "facname,address,boro,facsubgrp,latitude,longitude",
            "$where":  "facsubgrp='PUBLIC LIBRARIES'"},
    max_rows=500,
)
health_df = fetch_nyc_open_data(
    "ji82-xba5.json",
    params={"$select": "facname,address,boro,facsubgrp,latitude,longitude",
            "$where":  "facsubgrp IN ('HOSPITALS AND CLINICS','MENTAL HEALTH','OTHER HEALTH CARE')"},
    max_rows=5000,
)

def _add_point_layer(point_df, name, color, show=False):
    """Add a toggleable CircleMarker layer for points with latitude/longitude."""
    if point_df.empty:
        return
    point_df = point_df.copy()
    point_df["latitude"]  = pd.to_numeric(point_df["latitude"],  errors="coerce")
    point_df["longitude"] = pd.to_numeric(point_df["longitude"], errors="coerce")
    point_df = point_df.dropna(subset=["latitude", "longitude"])
    layer = folium.FeatureGroup(name=name, show=show)
    for _, row in point_df.iterrows():
        folium.CircleMarker(
            location=[float(row["latitude"]), float(row["longitude"])],
            radius=3, color=color, fill=True, fill_opacity=0.85,
            tooltip=row.get("facname","")
        ).add_to(layer)
    layer.add_to(final_map)

_add_point_layer(libraries_df, "Public libraries", "#7B241C", show=False)
_add_point_layer(health_df,    "Health facilities", "#117864", show=False)

# 311 heatmap toggle.
print("Re-fetching recent 311 points for the heatmap...")
from datetime import datetime, timedelta
cutoff = (datetime.utcnow() - timedelta(days=180)).strftime("%Y-%m-%dT00:00:00")
calls_df = fetch_nyc_open_data(
    "erm2-nwe9.json",
    params={"$select": "latitude,longitude",
            "$where":  f"created_date > '{cutoff}' AND latitude IS NOT NULL"},
    max_rows=20000,
)
if not calls_df.empty:
    calls_df["latitude"]  = pd.to_numeric(calls_df["latitude"],  errors="coerce")
    calls_df["longitude"] = pd.to_numeric(calls_df["longitude"], errors="coerce")
    heat_points = calls_df.dropna(subset=["latitude","longitude"])[
        ["latitude","longitude"]].values.tolist()
    heat_layer = folium.FeatureGroup(name="311 complaint density (heatmap)", show=False)
    HeatMap(heat_points, radius=8, blur=12, min_opacity=0.2).add_to(heat_layer)
    heat_layer.add_to(final_map)

# City-owned lot toggle — use PLUTO city_owned filter via the prepared tract field.
city_lot_tracts = scored[scored["city_owned_lots"].fillna(0) >= 1]
if not city_lot_tracts.empty:
    folium.GeoJson(
        city_lot_tracts,
        style_function=lambda feature: {"fillOpacity": 0,
                                        "color": "#E67E22", "weight": 1.5,
                                        "dashArray": "4, 3"},
        name="Tracts with ≥1 city-owned lot",
        show=False,
    ).add_to(final_map)

folium.LayerControl(collapsed=False).add_to(final_map)

final_path = os.path.join(OUTPUT_FOLDER, "maps", "site_selection_map.html")
final_map.save(final_path)
print(f"\nSaved final map: {final_path}")
display(final_map)

---

## Closing

A map with numbers on it is not a decision. It is a **proposal for how to see a problem**. Before this map goes into a presentation or a report, sit with these questions:

- **Who was in the room when the criteria were chosen?** If you wrote the criteria alone, the model encodes one person's framing of a community's needs. That is a starting point. It is not legitimacy.

- **Whose data is missing from this analysis?** Every absence in the data is a presence in someone's life that your map cannot see. Undocumented residents, people in informal housing, recent immigrants, day laborers, anyone who does not call 311 — they are in the city but not in this dataset.

- **What would change if the community being served had built this model instead of the designer?** That is a real methodological question. Co-production is not a label, it is a relationship. If the residents of your top tracts looked at this map, what would they push back on first?

The most useful thing this map can do is **start a conversation, not end one.** Bring it to a community board meeting. Bring it to a tenants' association. Bring it to the people whose tract scored highest, and ask them whether what you measured matches what they live. Then make a better map.
